# Math functions

0. Luminous Sinc Surface
1. Möbius strip
2. Klein bottle
3. Torus knot
4. Lorenz attractor
5. Gyroid surface
6. Mandelbulb-like field
7. Rössler attractor
8. Wave interference field
9. Superformula surfaces
10. Minimal surfaces (Schwarz / P-surface / Gyroid)

In [ ]:
# Rotating Luminous Sinc Surface v1
# Output: media-site/animations/Math/sinc_surface_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"   # webm | mp4 | gif
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 16, 9
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

COL = "#35f6ff"
BG = "black"


# -----------------------------------------------------------------------------
# Surface
# -----------------------------------------------------------------------------

N = 240

x = np.linspace(-12, 12, N)
y = np.linspace(-12, 12, N)

X, Y = np.meshgrid(x, y)

R = np.sqrt(X**2 + Y**2) + 1e-9

Z = np.sin(R) / R

# slight shaping
Z *= 2.2


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")

ax.set_facecolor(BG)

ax.set_xlim(-12, 12)
ax.set_ylim(-12, 12)
ax.set_zlim(-2.8, 2.8)

ax.set_box_aspect((1.8, 1.8, 0.7))

ax.grid(False)

# remove panes
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

# remove ticks
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

# transparent axis lines
ax.xaxis.line.set_color((0, 0, 0, 0))
ax.yaxis.line.set_color((0, 0, 0, 0))
ax.zaxis.line.set_color((0, 0, 0, 0))


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min())

    brightness = 0.22 + 0.95 * zn
    brightness *= (0.72 + 0.28 * pulse)

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = 0.92

    return np.clip(colors, 0, 1)


# -----------------------------------------------------------------------------
# Animation frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()

    ax.set_facecolor(BG)

    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_zlim(-2.8, 2.8)

    ax.set_box_aspect((1.8, 1.8, 0.7))

    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))

    # rotating camera
    azim = 360 * t
    elev = 32 + 8 * np.sin(tau * t)

    ax.view_init(
        elev=elev,
        azim=azim,
    )

    colors = build_colors(pulse)

    # glowing surface
    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=3,
        cstride=3,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
    )

    # wireframe overlay
    ax.plot_wireframe(
        X,
        Y,
        Z,
        rstride=12,
        cstride=12,
        color=COL,
        linewidth=0.35,
        alpha=0.22 + 0.18 * pulse,
    )

    # center glow ring
    theta = np.linspace(0, 2 * np.pi, 240)

    rr = 1.2 + 0.15 * pulse

    gx = rr * np.cos(theta)
    gy = rr * np.sin(theta)
    gz = np.sin(rr) / rr * np.ones_like(theta) * 1.02

    ax.plot(
        gx,
        gy,
        gz,
        color="white",
        linewidth=1.1,
        alpha=0.45 + 0.25 * pulse,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="sinc_surface_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

This surface is defined by the function:

$$ z = \frac{\sin(r)}{r} $$

where

$$ r = \sqrt{x^2 + y^2} $$

It is commonly known as:

* the sinc surface,
* the sombrero function,
* or an Airy-like wave surface.

Physical intuition:

* it represents a radially propagating wave;
* similar structures appear in diffraction patterns;
* signal processing;
* optics;
* radio physics;
* quantum mechanics;
* Fourier analysis.

Why it looks this way:

* the center contains the main maximum;
* concentric oscillating rings spread outward;
* the amplitude gradually decays approximately as 1/r.

# 1. Mobius strip

In [ ]:
# Möbius Strip — rotating mathematical sculpture
# Output: media-site/animations/Math/moebius_strip_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"


# -----------------------------------------------------------------------------
# Möbius geometry
# -----------------------------------------------------------------------------

u = np.linspace(0, 2 * np.pi, 320)
v = np.linspace(-0.45, 0.45, 80)

U, V = np.meshgrid(u, v)

R = 2.7

X = (R + V * np.cos(U / 2)) * np.cos(U)
Y = (R + V * np.cos(U / 2)) * np.sin(U)
Z = V * np.sin(U / 2)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")

ax.set_facecolor(BG)

ax.set_xlim(-4.2, 4.2)
ax.set_ylim(-4.2, 4.2)
ax.set_zlim(-2.4, 2.4)

ax.set_box_aspect((1, 1, 0.7))

ax.grid(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

ax.xaxis.line.set_color((0, 0, 0, 0))
ax.yaxis.line.set_color((0, 0, 0, 0))
ax.zaxis.line.set_color((0, 0, 0, 0))


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):

    zn = (Z - Z.min()) / (Z.max() - Z.min())

    brightness = 0.25 + 0.9 * zn
    brightness *= (0.72 + 0.28 * pulse)

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = 0.94

    return np.clip(colors, 0, 1)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()

    ax.set_facecolor(BG)

    ax.set_xlim(-4.2, 4.2)
    ax.set_ylim(-4.2, 4.2)
    ax.set_zlim(-2.4, 2.4)

    ax.set_box_aspect((1, 1, 0.7))

    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))

    # perfectly looping camera
    azim = 360 * t
    elev = 28 + 6 * np.sin(tau * t)

    ax.view_init(
        elev=elev,
        azim=azim,
    )

    colors = build_colors(pulse)

    # surface
    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
    )

    # glowing wireframe
    ax.plot_wireframe(
        X,
        Y,
        Z,
        rstride=6,
        cstride=12,
        color=COL,
        linewidth=0.35,
        alpha=0.22 + 0.18 * pulse,
    )

    # edge highlight
    edge_u = np.linspace(0, 2 * np.pi, 1200)
    edge_v = 0.45

    ex = (R + edge_v * np.cos(edge_u / 2)) * np.cos(edge_u)
    ey = (R + edge_v * np.cos(edge_u / 2)) * np.sin(edge_u)
    ez = edge_v * np.sin(edge_u / 2)

    ax.plot(
        ex,
        ey,
        ez,
        color="white",
        linewidth=1.0,
        alpha=0.35 + 0.25 * pulse,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="moebius_strip_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

# 2. Torus Knot

In [ ]:
# Torus Knot — rotating mathematical sculpture
# Output: media-site/animations/Math/torus_knot_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_DIM = "#1b7f88"
COL_GLOW = "#9ffcff"

P = 3
Q = 2

N = 1800
t_curve = np.linspace(0, 2 * np.pi, N)

R = 2.6
r = 0.85

X = (R + r * np.cos(Q * t_curve)) * np.cos(P * t_curve)
Y = (R + r * np.cos(Q * t_curve)) * np.sin(P * t_curve)
Z = r * np.sin(Q * t_curve)


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-4.4, 4.4)
    ax.set_ylim(-4.4, 4.4)
    ax.set_zlim(-3.2, 3.2)

    ax.set_box_aspect((1, 1, 0.75))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


setup_axes()


frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    azim = 360 * t
    elev = 30 + 7 * np.sin(tau * t)

    ax.view_init(elev=elev, azim=azim)

    # faint torus guide
    u = np.linspace(0, 2 * np.pi, 80)
    v = np.linspace(0, 2 * np.pi, 24)
    U, V = np.meshgrid(u, v)

    TX = (R + r * np.cos(V)) * np.cos(U)
    TY = (R + r * np.cos(V)) * np.sin(U)
    TZ = r * np.sin(V)

    ax.plot_wireframe(
        TX,
        TY,
        TZ,
        rstride=4,
        cstride=6,
        color=COL_DIM,
        linewidth=0.22,
        alpha=0.08 + 0.08 * pulse,
    )

    # main glowing knot
    ax.plot(
        X,
        Y,
        Z,
        color=COL_GLOW,
        linewidth=4.0,
        alpha=0.16 + 0.12 * pulse,
    )

    ax.plot(
        X,
        Y,
        Z,
        color=COL,
        linewidth=1.45,
        alpha=0.86,
    )

    # moving highlight point
    idx = int((N - 1) * t)
    hx, hy, hz = X[idx], Y[idx], Z[idx]

    ax.scatter(
        [hx],
        [hy],
        [hz],
        s=48,
        color="white",
        alpha=0.55 + 0.35 * pulse,
        depthshade=False,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="torus_knot_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

The torus knot is a closed curve wrapped around the surface of a torus. In this version it uses a (3, 2) knot: the curve winds three times around the main axis of the torus and twice through its tube before closing perfectly. This makes it a natural mathematical symbol of periodicity, resonance, orbital structure, and cyclic motion.

# 3. Lorenz Attractor

In [ ]:
# Lorenz Attractor — rotating mathematical sculpture
# Output: media-site/animations/Math/lorenz_attractor_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GLOW = "#9ffcff"
COL_DIM = "#1b7f88"


# -----------------------------------------------------------------------------
# Lorenz system
# -----------------------------------------------------------------------------

SIGMA = 10.0
RHO = 28.0
BETA = 8.0 / 3.0

DT = 0.005
STEPS = 16000

xyz = np.empty((STEPS, 3), dtype=np.float64)
xyz[0] = [0.1, 1.0, 1.05]

for i in range(1, STEPS):
    x, y, z = xyz[i - 1]

    dx = SIGMA * (y - x)
    dy = x * (RHO - z) - y
    dz = x * y - BETA * z

    xyz[i] = xyz[i - 1] + DT * np.array([dx, dy, dz])


# remove transient
xyz = xyz[2500:]

X = xyz[:, 0]
Y = xyz[:, 1]
Z = xyz[:, 2]

# normalize / center
X = X / 10.0
Y = Y / 10.0
Z = (Z - 25.0) / 10.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-2.7, 2.7)
    ax.set_ylim(-3.2, 3.2)
    ax.set_zlim(-2.2, 2.4)

    ax.set_box_aspect((1, 1, 0.85))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


setup_axes()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

n = len(X)

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    # seamless camera rotation
    azim = 360 * t
    elev = 26 + 6 * np.sin(tau * t)

    ax.view_init(elev=elev, azim=azim)

    # faint full attractor
    ax.plot(
        X,
        Y,
        Z,
        color=COL_DIM,
        linewidth=0.45,
        alpha=0.18,
    )

    # animated active trail
    head = int(n * t)
    trail_len = 2200

    idx = (np.arange(head - trail_len, head) % n).astype(int)

    tx = X[idx]
    ty = Y[idx]
    tz = Z[idx]

    # glow layer
    ax.plot(
        tx,
        ty,
        tz,
        color=COL_GLOW,
        linewidth=3.2,
        alpha=0.10 + 0.08 * pulse,
    )

    # core line
    ax.plot(
        tx,
        ty,
        tz,
        color=COL,
        linewidth=1.0,
        alpha=0.85,
    )

    # moving point
    hx, hy, hz = X[head % n], Y[head % n], Z[head % n]

    ax.scatter(
        [hx],
        [hy],
        [hz],
        s=42,
        color="white",
        alpha=0.55 + 0.35 * pulse,
        depthshade=False,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="lorenz_attractor_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

The Lorenz attractor is a classic chaotic system. It comes from a simplified model of atmospheric convection. Its trajectory never settles into a simple orbit, yet it remains trapped inside a recognizable butterfly-shaped structure. It is often used as a visual symbol of deterministic chaos: the rules are exact, but long-term behavior is extremely sensitive to initial conditions.

# 4. Rössler Attractor

In [ ]:
# Rössler Attractor — rotating mathematical sculpture
# Output: media-site/animations/Math/rossler_attractor_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GLOW = "#9ffcff"
COL_DIM = "#1b7f88"


# -----------------------------------------------------------------------------
# Rössler system
# -----------------------------------------------------------------------------

A = 0.2
B = 0.2
C = 5.7

DT = 0.01
STEPS = 22000

xyz = np.empty((STEPS, 3), dtype=np.float64)
xyz[0] = [0.1, 0.0, 0.0]

for i in range(1, STEPS):
    x, y, z = xyz[i - 1]

    dx = -y - z
    dy = x + A * y
    dz = B + z * (x - C)

    xyz[i] = xyz[i - 1] + DT * np.array([dx, dy, dz])


# remove transient
xyz = xyz[4000:]

X = xyz[:, 0]
Y = xyz[:, 1]
Z = xyz[:, 2]

# normalize / center
X = (X - X.mean()) / 7.0
Y = (Y - Y.mean()) / 7.0
Z = (Z - Z.mean()) / 7.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-2.4, 2.4)
    ax.set_zlim(-1.5, 2.2)

    ax.set_box_aspect((1, 1, 0.8))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


setup_axes()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

n = len(X)

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    azim = 360 * t
    elev = 28 + 6 * np.sin(tau * t)

    ax.view_init(elev=elev, azim=azim)

    # faint full attractor
    ax.plot(
        X,
        Y,
        Z,
        color=COL_DIM,
        linewidth=0.45,
        alpha=0.20,
    )

    # active moving trail
    head = int(n * t)
    trail_len = 2600

    idx = (np.arange(head - trail_len, head) % n).astype(int)

    tx = X[idx]
    ty = Y[idx]
    tz = Z[idx]

    ax.plot(
        tx,
        ty,
        tz,
        color=COL_GLOW,
        linewidth=3.0,
        alpha=0.10 + 0.08 * pulse,
    )

    ax.plot(
        tx,
        ty,
        tz,
        color=COL,
        linewidth=1.0,
        alpha=0.88,
    )

    hx, hy, hz = X[head % n], Y[head % n], Z[head % n]

    ax.scatter(
        [hx],
        [hy],
        [hz],
        s=42,
        color="white",
        alpha=0.55 + 0.35 * pulse,
        depthshade=False,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="rossler_attractor_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

The Rössler attractor is a chaotic dynamical system introduced as a simpler alternative to the Lorenz system. Its trajectory forms a spiral-like structure with occasional excursions away from the central loop. The system is deterministic, but its long-term behavior is chaotic, making it a useful visual metaphor for ordered motion gradually turning into unpredictability.

# 5. Gyroid Surface

In [7]:
# Gyroid Surface — rotating mathematical sculpture
# Output: media-site/animations/Math/gyroid_surface_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_DIM = "#1b7f88"


N = 42
L = 2.4 * np.pi

x = np.linspace(-L, L, N)
y = np.linspace(-L, L, N)
z = np.linspace(-L, L, N)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

F = (
    np.sin(X) * np.cos(Y)
    + np.sin(Y) * np.cos(Z)
    + np.sin(Z) * np.cos(X)
)

verts, faces, normals, values = measure.marching_cubes(
    F,
    level=0.0,
    spacing=(
        (x.max() - x.min()) / (N - 1),
        (y.max() - y.min()) / (N - 1),
        (z.max() - z.min()) / (N - 1),
    ),
)

# center mesh around origin
verts[:, 0] -= verts[:, 0].mean()
verts[:, 1] -= verts[:, 1].mean()
verts[:, 2] -= verts[:, 2].mean()

# normalize scale
verts /= np.max(np.abs(verts)) / 3.2


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)
    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


frames = []
tau = 2 * np.pi

mesh_faces = verts[faces]

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    azim = 360 * t
    elev = 28 + 6 * np.sin(tau * t)

    ax.view_init(elev=elev, azim=azim)

    alpha = 0.32 + 0.18 * pulse

    face_color = (
        base_rgb[0],
        base_rgb[1],
        base_rgb[2],
        alpha,
    )

    mesh = Poly3DCollection(
        mesh_faces,
        facecolor=face_color,
        edgecolor=COL_DIM,
        linewidth=0.0,
        alpha=alpha,
    )

    ax.add_collection3d(mesh)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="gyroid_surface_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/gyroid_surface_v1.webm
Frames: 192
Size: 1604.6 KB


[out#0/webm @ 0x12bf06240] video:1603KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.116562%
frame=  192 fps= 31 q=32.0 Lsize=    1605KiB time=00:00:08.00 bitrate=1643.1kbits/s speed=1.27x    


A gyroid is a triply periodic minimal surface. It repeats in three spatial directions and has no straight lines or mirror symmetry. It is important in mathematics, materials science, crystallography, and soft matter physics. Similar structures appear in block copolymers, biological membranes, photonic materials, and some porous crystals. Its visual form is a continuous labyrinth-like surface dividing space into two intertwined regions.

# 6. Wave Interference Field

In [8]:
# Wave Interference Field — rotating mathematical sculpture
# Output: media-site/animations/Math/wave_interference_field_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"
COL = "#35f6ff"

N = 180

x = np.linspace(-7, 7, N)
y = np.linspace(-7, 7, N)
X, Y = np.meshgrid(x, y)

sources = [
    (-2.8, -1.4, 1.00),
    (2.4, -0.8, 0.85),
    (0.3, 2.7, 0.75),
]


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-7, 7)
    ax.set_ylim(-7, 7)
    ax.set_zlim(-2.8, 2.8)

    ax.set_box_aspect((1, 1, 0.45))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_surface(phase: float) -> np.ndarray:
    Z = np.zeros_like(X)

    for sx, sy, amp in sources:
        R = np.sqrt((X - sx) ** 2 + (Y - sy) ** 2) + 1e-6
        Z += amp * np.sin(2.8 * R - phase) / (1.0 + 0.28 * R)

    return Z


def build_colors(Z: np.ndarray, pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = 0.92

    return np.clip(colors, 0, 1)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)
    wave_phase = tau * 2 * t

    Z = build_surface(wave_phase)
    colors = build_colors(Z, pulse)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    azim = 360 * t
    elev = 34 + 5 * np.sin(tau * t)

    ax.view_init(elev=elev, azim=azim)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=3,
        cstride=3,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
    )

    ax.plot_wireframe(
        X,
        Y,
        Z,
        rstride=12,
        cstride=12,
        color=COL,
        linewidth=0.35,
        alpha=0.18 + 0.16 * pulse,
    )

    # source glow points
    for sx, sy, _ in sources:
        ax.scatter(
            [sx],
            [sy],
            [1.45],
            s=36,
            color="white",
            alpha=0.45 + 0.3 * pulse,
            depthshade=False,
        )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="wave_interference_field_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/wave_interference_field_v1.webm
Frames: 192
Size: 2018.9 KB


[out#0/webm @ 0x127e25b10] video:2017KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.093443%
frame=  192 fps= 36 q=32.0 Lsize=    2019KiB time=00:00:08.00 bitrate=2067.4kbits/s speed=1.52x    


This visualization shows an interference field formed by several radial wave sources. Each source emits circular waves, and the final surface is the sum of those waves. Where wave crests meet, the height increases; where crests and troughs meet, they cancel. This is the same basic principle behind water ripples, sound interference, diffraction, radio waves, and quantum wave patterns.

# 7.Superformula Surface

In [9]:
# Superformula Surface — rotating mathematical sculpture
# Output: media-site/animations/Math/superformula_surface_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"
COL = "#35f6ff"

U = np.linspace(0, 2 * np.pi, 220)
V = np.linspace(-np.pi / 2, np.pi / 2, 120)
U, V = np.meshgrid(U, V)


def superformula(phi, m, n1, n2, n3, a=1.0, b=1.0):
    t1 = np.abs(np.cos(m * phi / 4) / a) ** n2
    t2 = np.abs(np.sin(m * phi / 4) / b) ** n3
    r = (t1 + t2) ** (-1 / n1)
    return np.nan_to_num(r, nan=0.0, posinf=0.0, neginf=0.0)


r1 = superformula(U, m=7, n1=0.35, n2=1.7, n3=1.7)
r2 = superformula(V, m=4, n1=0.45, n2=1.2, n3=1.2)

X = 2.7 * r1 * np.cos(U) * r2 * np.cos(V)
Y = 2.7 * r1 * np.sin(U) * r2 * np.cos(V)
Z = 2.7 * r2 * np.sin(V)


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.yaxis.pane.set_edgecolor((0, 0, 0, 0))
    ax.zaxis.pane.set_edgecolor((0, 0, 0, 0))

    ax.xaxis.line.set_color((0, 0, 0, 0))
    ax.yaxis.line.set_color((0, 0, 0, 0))
    ax.zaxis.line.set_color((0, 0, 0, 0))


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.2 + 0.95 * zn
    brightness *= 0.72 + 0.28 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = 0.93

    return np.clip(colors, 0, 1)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
    )

    ax.plot_wireframe(
        X,
        Y,
        Z,
        rstride=8,
        cstride=10,
        color=COL,
        linewidth=0.3,
        alpha=0.18 + 0.15 * pulse,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())


plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="superformula_surface_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/superformula_surface_v1.webm
Frames: 192
Size: 312.5 KB


The superformula is a generalization of the superellipse. By changing a few parameters, it can generate a wide range of organic and geometric forms: flowers, starfish-like shapes, rounded polygons, shells, and abstract biological structures. In 3D, two superformula profiles can be combined to create a sculptural surface with radial symmetry.